# Enterprise RAG — Hands-On, Part 2 of 11: The policy engine

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
from enterprise_rag.ingest.loader import load_corpus
from enterprise_rag.identity import list_principals, get_principal

docs = load_corpus()

---
# Part 2 - The policy engine

This runs **before** any retrieval and involves **no LLM at all**. Access control is decided
statically and is never delegated to the model.

`decide(principal, resource)` returns an allow/deny, the **named rule** that decided it, and any
**obligations** attached to an allow (e.g. "you may read this, but with PII redacted").

In [ ]:
from enterprise_rag.authz.policy import decide

TODAY = "2026-08-22"
by_id = {d.attrs.doc_id: d.attrs for d in docs}

def check(user_id, doc_id, as_of=TODAY):
    from enterprise_rag.identity import get_principal
    p = get_principal(user_id)
    d = decide(p, by_id[doc_id], {"as_of": as_of})
    verdict = "ALLOW" if d.allowed else "DENY "
    obl = f"  obligations={d.obligations}" if d.obligations else ""
    print(f"{verdict}  {p.role:<38} -> {doc_id:<15} [{d.rule}] {d.reason}{obl}")

# The same confidential post-mortem, seen by five different people.
for uid in ["u_marco_t3", "u_lena_t1", "u_sofia_am", "u_jin_us_t3", "u_attacker_other_tenant"]:
    print(uid,end = "  ")
    check(uid, "PM-2026-03-14")

u_marco_t3  ALLOW  Tier 3 Escalation Engineer             -> PM-2026-03-14   [group_membership] group overlap: engineering, support-tier3  obligations=['audit_access']
u_lena_t1  DENY   Tier 1 Support Agent                   -> PM-2026-03-14   [clearance] resource is 'confidential' but principal clearance is 'internal'
u_sofia_am  DENY   Enterprise Account Manager             -> PM-2026-03-14   [default_deny] no grant matched; the default is deny
u_jin_us_t3  DENY   Tier 3 Escalation Engineer (US region) -> PM-2026-03-14   [data_residency] resource is locked to region 'EU', principal is in 'US'
u_attacker_other_tenant  DENY   Support Agent                          -> PM-2026-03-14   [tenant_isolation] principal tenant 'acme' != resource tenant 'meridian'


Read those denial reasons carefully - each one is a *different rule*:

- **Lena** (Tier 1) - `clearance`: the document outranks her.
- **Sofia** (Account Manager) - `default_deny`: high enough clearance, but no group grants it.
- **Jin** (Tier 3, **US**) - `data_residency`: identical role and clearance to Marco, wrong region.
  This is a real contractual term in the Vertex MSA.
- **Other tenant** - `tenant_isolation`: every group, top clearance, still nothing.

That last one is the property worth stressing: **no combination of privileges crosses a tenant
boundary.**

In [ ]:
# Obligations: an ALLOW can come with conditions attached.
check("u_lena_t1", "TK-4471")          # entitled to PII -> no obligation
check("u_tom_contractor", "TK-4488")   # contractor -> redact_pii
check("u_sofia_am", "CT-VTX-001")      # confidential -> audit_access

ALLOW  Tier 1 Support Agent                   -> TK-4471         [group_membership] group overlap: support-tier1
ALLOW  Tier 1 Support Agent (external contractor) -> TK-4488         [group_membership] group overlap: support-tier1  obligations=['redact_pii']
ALLOW  Enterprise Account Manager             -> CT-VTX-001      [group_membership] group overlap: account-management, sales  obligations=['audit_access']


### Time-bound and compartmented access

Clearance alone is never enough. `SA-2026-07` is a security advisory that is both **embargoed until
2026-09-01** and restricted to the **vuln-response** compartment.

In [ ]:
print("SA-2026-07: restricted, embargoed until 2026-09-01, need-to-know=['vuln-response']\n")
for uid in ["u_ravi_sec", "u_erin_secmgr"]:
    for as_of in ["2026-08-22", "2026-09-02"]:
        print(f"  as of {as_of}: ", end="")
        check(uid, "SA-2026-07", as_of=as_of)

SA-2026-07: restricted, embargoed until 2026-09-01, need-to-know=['vuln-response']

  as of 2026-08-22: DENY   Security Engineer                      -> SA-2026-07      [embargo] resource is embargoed until 2026-09-01 (today is 2026-08-22)
  as of 2026-09-02: ALLOW  Security Engineer                      -> SA-2026-07      [group_membership] group overlap: security  obligations=['audit_access']
  as of 2026-08-22: DENY   Security Manager (no vuln-response compartment) -> SA-2026-07      [embargo] resource is embargoed until 2026-09-01 (today is 2026-08-22)
  as of 2026-09-02: DENY   Security Manager (no vuln-response compartment) -> SA-2026-07      [need_to_know] principal lacks need-to-know compartment(s): vuln-response


Ravi has the compartment, so he gains access the moment the embargo lifts. Erin has the *same
restricted clearance* but no compartment, so the date never helps her.

This is precisely the kind of rule that is one line in ABAC and a combinatorial mess in plain RBAC.

### The full visibility matrix

Computed by the policy engine alone. This table *is* the security specification - it can be reviewed
by someone who cannot read Python.

In [ ]:
from enterprise_rag.identity import get_principal

principals = list_principals()
hdr = f"{'document':<16}{'source':<12}{'sens':<13}"
for p in principals:
    hdr += f"{p.user_id.replace('u_','')[:8]:>10}"
print(hdr); print("-" * len(hdr))

for a in sorted(by_id.values(), key=lambda x: (x.source, x.doc_id)):
    row = f"{a.doc_id:<16}{a.source:<12}{a.sensitivity:<13}"
    for p in principals:
        row += f"{('Y' if decide(p, a, {'as_of': TODAY}).allowed else '.'):>10}"
    print(row)
print("\nY = readable    . = denied")

document        source      sens            lena_t1  marco_t3  sofia_am  ravi_sec  erin_sec  tom_cont  dana_ext  jin_us_t  attacker
-----------------------------------------------------------------------------------------------------------------------------------
SA-2026-05      advisory    restricted            .         .         .         Y         .         .         .         .         .
SA-2026-07      advisory    restricted            .         .         .         .         .         .         .         .         .
CT-KST-003      contract    confidential          .         .         Y         .         .         .         .         .         .
CT-NGR-002      contract    confidential          .         .         .         .         .         .         .         .         .
CT-VTX-001      contract    confidential          .         .         Y         .         .         .         .         .         .
HC-001          helpcenter  public                Y         Y         Y     

---

**◀ Previous:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb)

**Next ▶:** [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb)
